<a href="https://colab.research.google.com/github/sittana-afifi/AIMS-KTT-Hackathon---S2.T3.1-AI-Math-Tutor-for-Early-Learners/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Environment Setup**

In [1]:
# Install specialized CPU-optimized libraries
# 1. LLM Engine (New stable version for GGUF)
!pip install llama-cpp-python==0.2.76

# 2. ASR Engine (Whisper)
!pip install faster-whisper

# 3. Audio & UI Utilities
!pip install librosa soundfile gradio

# 4. Math & Downloading Tools
!pip install "numpy<2.0.0" scipy huggingface_hub

#5. Core engines
!pip install --force-reinstall faster-whisper llama-cpp-python



  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached llama_cpp_python-0.3.20.tar.gz (59.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ctranslate2-4.7.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached onnxruntime-1.25.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
  Using cached av-17.0.1-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (4.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x8

**Download and Initialize the LLM (Mwenyura-Nano)**

In [1]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import os

# 1. Download the specific 2-bit quantized file (Tiny!)
# This stays close to our 75MB target for the model itself
model_path = hf_hub_download(
    repo_id="Qwen/Qwen1.5-0.5B-Chat-GGUF",
    filename="qwen1_5-0_5b-chat-q2_k.gguf"
)

print(f"Model downloaded to: {model_path}")

# 2. Initialize the Llama Engine
# n_ctx=512 keeps memory usage low for mobile/edge hardware
llm = Llama(
    model_path=model_path,
    n_ctx=512,
    n_threads=2, # Optimized for dual-core CPUs
    verbose=False
)

print("✅ Mwenyura AI LLM Engine Initialized successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model downloaded to: /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat-GGUF/snapshots/cfab082d2fef4a8736ef384dc764c2fb6887f387/qwen1_5-0_5b-chat-q2_k.gguf


llama_context: n_ctx_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Mwenyura AI LLM Engine Initialized successfully!


**Initialize the ASR Engine (Whisper-Tiny)**

In [2]:
from faster_whisper import WhisperModel

# Load Whisper-Tiny in 8-bit quantization
# This model is ~39MB, fitting well within our footprint
asr_model = WhisperModel("tiny", device="cpu", compute_type="int8")

def transcribe_child_voice(audio_file):
    """
    Converts audio input to text.
    Beam_size=5 helps with the high-pitch variations in child speech.
    """
    segments, info = asr_model.transcribe(audio_file, beam_size=5)
    text = " ".join([segment.text for segment in segments])
    return text.strip()

print("✅ Mwenyura AI ASR Engine Initialized!")

✅ Mwenyura AI ASR Engine Initialized!


**The Bayesian Knowledge Tracing (BKT) Brain**

In [3]:
import numpy as np

class MwenyuraBrain:
    def __init__(self):
        # We track different skills separately
        self.skills = {
            "counting": 0.25,      # Initial probability (P_L0): 25% chance they know it
            "addition": 0.10,      # P_L0 for addition
            "subtraction": 0.05    # P_L0 for subtraction
        }

        # BKT Standard Parameters (Used in educational AI)
        self.p_transit = 0.1  # Probability of learning the skill during a turn
        self.p_slip = 0.1     # Probability of knowing it but making a mistake (10%)
        self.p_guess = 0.25    # Probability of not knowing it but guessing right (25%)

    def update_mastery(self, skill, is_correct):
        """
        Updates the mastery probability using Bayesian inference.
        """
        old_mastery = self.skills.get(skill, 0.1)

        if is_correct:
            # Probability they knew it, given they got it right
            p_known_given_obs = (old_mastery * (1 - self.p_slip)) / \
                                (old_mastery * (1 - self.p_slip) + (1 - old_mastery) * self.p_guess)
        else:
            # Probability they knew it, given they got it wrong
            p_known_given_obs = (old_mastery * self.p_slip) / \
                                (old_mastery * self.p_slip + (1 - old_mastery) * (1 - self.p_guess))

        # Update mastery: Known given observation + newly learned probability
        new_mastery = p_known_given_obs + (1 - p_known_given_obs) * self.p_transit

        # Clip to ensure it stays between 0 and 1
        self.skills[skill] = np.clip(new_mastery, 0, 0.99)
        return self.skills[skill]

    def get_status(self, skill):
        prob = self.skills.get(skill, 0.0)
        if prob > 0.85: return "Mastered"
        if prob > 0.50: return "Learning"
        return "Struggling"

# Initialize the brain
brain = MwenyuraBrain()
print("✅ Knowledge Tracing Brain is ready.")

✅ Knowledge Tracing Brain is ready.


**The "Differential Privacy" Sync logic**

In [4]:
def get_private_mastery(skill):
    """
    Adds Laplacian noise to the score before 'syncing' to a central server.
    This protects the child's exact performance from being exposed.
    """
    actual_score = brain.skills.get(skill, 0.0)

    # Epsilon is the 'privacy budget'. Lower means more noise.
    epsilon = 0.5
    noise = np.random.laplace(0, 1/epsilon) * 0.05 # Scaled noise

    private_score = np.clip(actual_score + noise, 0, 1)
    return private_score

print(f"Actual: {brain.skills['counting']} | Private Sync: {get_private_mastery('counting')}")

Actual: 0.25 | Private Sync: 0.31644400448756227


**The Final Integration (Initial Demo)**

In [13]:
import gradio as gr
import numpy as np
import random
import os
from llama_cpp import Llama
from faster_whisper import WhisperModel
from huggingface_hub import hf_hub_download

# ==========================================
# 1. THE BRAIN: Bayesian Knowledge Tracing
# ==========================================
class MwenyuraBrain:
    def __init__(self):
        # p_L0: Initial Mastery, p_T: Transition, p_S: Slip, p_G: Guess
        self.skills = {"counting": 0.25}
        self.p_transit = 0.1
        self.p_slip = 0.1
        self.p_guess = 0.25

    def update_mastery(self, skill, is_correct):
        old_L = self.skills.get(skill, 0.25)
        if is_correct:
            p_known_obs = (old_L * (1 - self.p_slip)) / (old_L * (1 - self.p_slip) + (1 - old_L) * self.p_guess)
        else:
            p_known_obs = (old_L * self.p_slip) / (old_L * self.p_slip + (1 - old_L) * (1 - self.p_guess))

        new_L = p_known_obs + (1 - p_known_obs) * self.p_transit
        self.skills[skill] = np.clip(new_L, 0, 0.99)
        return self.skills[skill]

    def get_status(self, skill):
        prob = self.skills.get(skill, 0.0)
        if prob > 0.80: return "🌟 Mastered"
        if prob > 0.45: return "📖 Learning"
        return "🌱 Struggling"

brain = MwenyuraBrain()

# ==========================================
# 2. THE ENGINES: AI Initialization
# ==========================================
print("📥 Loading Mwenyura Engines (Safe Mode)...")
# Download Qwen 0.5B (Quantized for efficiency)
model_path = hf_hub_download(repo_id="Qwen/Qwen1.5-0.5B-Chat-GGUF", filename="qwen1_5-0_5b-chat-q2_k.gguf")
llm = Llama(model_path=model_path, n_ctx=512, n_threads=2, verbose=False)

# Load Whisper Tiny for English
asr_model = WhisperModel("tiny.en", device="cpu", compute_type="int8")

# ==========================================
# 3. CORE LOGIC FUNCTIONS
# ==========================================
def generate_tutor_response(is_correct):
    """Provides culturally relevant feedback with Kinyarwanda scaffolding."""
    if is_correct:
        return random.choice([
            "Ni byiza cyane! Correct! Good job!",
            "Perfect! You are a math star! ⭐",
            "That's right! You are doing amazing.",
            "Urashoboye! (You are capable!) 1 + 1 is 2!"
        ])
    else:
        return random.choice([
            "Gerageza nabone! Let's try again.",
            "Keep trying! I know you can get it.",
            "Not quite, but don't give up!",
            "Let's count together one more time."
        ])

def mwenyura_tutor_loop(audio_path):
    if audio_path is None:
        return "Please record your voice!", "25%", "🌱 Struggling"

    try:
        # A. Listen (ASR)
        segments, _ = asr_model.transcribe(audio_path, beam_size=5)
        text = " ".join([s.text for s in segments]).strip().lower()
        print(f"DEBUG - Heard: '{text}'")

        # B. Noise/Grice Filter
        if text == "grice" or len(text) < 2:
            return "I didn't quite catch that. Could you say it again clearly?", f"{round(brain.skills['counting'] * 100)}%", brain.get_status("counting")

        # C. Logic Check (Handles phonetic misinterpretations)
        is_correct = any(word in text for word in ["2", "two", "to", "too", "ibiri"])

        # D. Update Brain (BKT)
        new_mastery = brain.update_mastery("counting", is_correct)
        status = brain.get_status("counting")

        # E. Get Friendly Response
        response = generate_tutor_response(is_correct)

        return response, f"{round(new_mastery * 100)}%", status
    except Exception as e:
        return f"Error: {e}", "0%", "Error"

# ==========================================
# 4. THE GRADIO UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☀️ Mwenyura AI: Math Tutor (Tier 3)")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🎤 Step 1: Speak to Mwenyura")
            audio_input = gr.Audio(sources="microphone", type="filepath", label="Record Answer")
            with gr.Row():
                submit_btn = gr.Button("✅ Check Answer", variant="primary")
                reset_btn = gr.Button("🔄 Reset App")

        with gr.Column():
            gr.Markdown("### 🤖 Step 2: Mwenyura's Feedback")
            output_text = gr.Textbox(
                label="Mwenyura says:",
                value="Muraho! I am Mwenyura. Let's count! How many is 1 + 1?",
                interactive=False
            )
            with gr.Row():
                mastery_bar = gr.Label(value="25%", label="Mastery Probability")
                status_tag = gr.Label(value="🌱 Struggling", label="Learning Status")

    # --- TEACHER DASHBOARD SECTION ---
    with gr.Accordion("📊 Teacher Dashboard (Insights)", open=False):
        gr.Markdown("### Student Performance Analysis")
        report_btn = gr.Button("Generate Student Report")
        report_out = gr.Markdown("Click to see progress data.")

    def generate_report():
        score = brain.skills['counting']
        status = brain.get_status('counting')
        stars = "⭐" * int(score * 5 + 1)
        return f"""
        **Student Mastery Report**
        - **Current Skill:** Number Sense (Counting)
        - **Mathematical Probability:** {int(score*100)}%
        - **Proficiency Level:** {status}
        - **Visual Progress:** {stars}

        **Tutor Recommendation:** {"Ready for Level 2: Simple Subtraction." if score > 0.8 else "Continue reinforcement with Level 1 counting."}
        """

    # Button Functionality
    submit_btn.click(
        mwenyura_tutor_loop,
        inputs=[audio_input],
        outputs=[output_text, mastery_bar, status_tag]
    )

    reset_btn.click(
        lambda: (None, "Muraho! I am Mwenyura. Let's count! How many is 1 + 1?", "25%", "🌱 Struggling"),
        outputs=[audio_input, output_text, mastery_bar, status_tag]
    )

    report_btn.click(generate_report, outputs=report_out)

# ==========================================
# 5. LAUNCH
# ==========================================
if __name__ == "__main__":
    # share=True is vital for Microphone access in Colab
    demo.launch(share=True, debug=True)

📥 Loading Mwenyura Engines (Safe Mode)...


llama_context: n_ctx_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
/tmp/ipykernel_28514/3051776794.py:101: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4eee69bebfc0deada2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


DEBUG - Heard: 'two'
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4eee69bebfc0deada2.gradio.live


**Demo with provided seed**

In [36]:
import gradio as gr
import numpy as np
import json
import os
from faster_whisper import WhisperModel

# ==========================================
# 1. SEED LOADER & IMAGE MAPPER
# ==========================================
class SeedManager:
    def __init__(self):
        self.curriculum = self._load_curriculum()
        self.schema = self._load_schema()

    def _load_curriculum(self):
        paths = ['/content/seed/curriculum_seed.json', 'curriculum_seed.json', 'seed/curriculum_seed.json']
        for path in paths:
            if os.path.exists(path):
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    return [{"id": i.get("id"), "skill": i.get("skill", "counting"),
                             "question": i.get("stem_en"), "visual": i.get("visual"),
                             "answer": str(i.get("answer_int"))} for i in data]
        return [{"question": "1+1?", "answer": "2", "skill": "addition", "visual": None}]

    def _load_schema(self):
        if os.path.exists('/content/seed/parent_report_schema.json'):
            with open('/content/seed/parent_report_schema.json', 'r') as f: return json.load(f)
        return {"learner_id": "Mwenyura_Student"}

seeds = SeedManager()

# ==========================================
# 2. THE BRAIN
# ==========================================
class MwenyuraBrain:
    def __init__(self):
        self.current_idx = 0
        self.skill_mastery = {
            "counting": 0.45, "addition": 0.30, "subtraction": 0.20,
            "word_problem": 0.20, "number_sense": 0.20
        }

    def update(self, is_correct):
        skill = seeds.curriculum[self.current_idx].get("skill", "counting")
        if is_correct:
            self.skill_mastery[skill] = min(0.99, self.skill_mastery[skill] + 0.15)
            if self.current_idx < len(seeds.curriculum) - 1:
                self.current_idx += 1
                return True
        else:
            self.skill_mastery[skill] = max(0.1, self.skill_mastery[skill] - 0.05)
        return False

    def get_avg(self):
        return int(np.mean(list(self.skill_mastery.values())) * 100)

brain = MwenyuraBrain()
asr_model = WhisperModel("tiny.en", device="cpu", compute_type="int8")

# ==========================================
# 3. INTERACTION & IMAGE LOCATOR
# ==========================================
NUM_WORDS = {"1": ["one", "rimwe"], "2": ["two", "deux", "kabiri"], "3": ["three", "trois", "gatatu"], "4": ["four", "quatre", "kane"], "5": ["five", "cinq", "gatanu"]}

def find_image(visual_name):
    if not visual_name: return None
    # Add mapping for your specific uploaded files
    image_map = {"apples_3": "image_98439c.png", "goats_5": "image_97d300.png"}
    mapped_name = image_map.get(visual_name, visual_name)
    for loc in ["/content/seed/", "/content/", "./"]:
        for ext in ['', '.png', '.jpg']:
            path = os.path.join(loc, f"{mapped_name}{ext}")
            if os.path.exists(path): return path
    return None

def tutor_step(audio_path):
    item = seeds.curriculum[brain.current_idx]
    if not audio_path:
        return None, f"🔇 I didn't hear you! \n\n{item['question']}", f"{brain.get_avg()}%", find_image(item['visual'])

    try:
        segments, _ = asr_model.transcribe(audio_path)
        heard = " ".join([s.text for s in segments]).strip().lower()
        is_correct = (item["answer"] in heard) or any(w in heard for w in NUM_WORDS.get(item["answer"], []))
        did_advance = brain.update(is_correct)
        new_item = seeds.curriculum[brain.current_idx]
        msg = f"✅ Correct! '{heard}' \n\nNEXT: {new_item['question']}" if is_correct else f"❌ Try again! '{heard}' \n\nSTILL ON: {item['question']}"
        return None, msg, f"{brain.get_avg()}%", find_image(new_item['visual'])
    except Exception as e: return None, f"Error: {e}", "0%", None

# ==========================================
# 4. THE ORGANIZED TABLE REPORT
# ==========================================
def generate_parent_report():
    learner = seeds.schema.get('learner_id', 'Student')
    avg = brain.get_avg()
    best_skill = max(brain.skill_mastery, key=brain.skill_mastery.get)
    weak_skill = min(brain.skill_mastery, key=brain.skill_mastery.get)

    # Table Header
    report = f"# 📊 Progress Report for {learner}\n"
    report += f"**Overall Mastery:** {avg}% {'⭐' * (avg // 20)}\n\n"
    report += "| Skill Area | Progress Status | Mastery Level |\n"
    report += "| :--- | :--- | :--- |\n"

    # Table Rows
    for skill, val in brain.skill_mastery.items():
        bar = "🟩" * max(1, int(val * 10)) + "⬜" * (10 - max(1, int(val * 10)))
        report += f"| {skill.title().replace('_', ' ')} | {bar} | {int(val*100)}% |\n"

    # Recommendation Section
    report += f"\n---\n"
    report += f"### 💡 Personalized Recommendation\n"
    report += f"**Strengths:** Your child is doing excellent in **{best_skill.title()}**! 🌟\n\n"
    report += f"**Action Plan:** We noticed **{weak_skill.replace('_', ' ')}** is a bit challenging. "
    report += f"We recommend focusing on daily 5-minute verbal counting games to boost confidence in this area."

    return report

# ==========================================
# 5. UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☀️ Mwenyura AI Math Tutor")
    with gr.Row():
        with gr.Column():
            q_image = gr.Image(value=find_image(seeds.curriculum[0]['visual']), label="Visual Aid")
            audio_in = gr.Audio(sources="microphone", type="filepath")
            btn = gr.Button("Submit Answer", variant="primary")
        with gr.Column():
            output_text = gr.Textbox(label="Tutor", value=f"Muraho! {seeds.curriculum[0]['question']}", lines=6)
            m_lab = gr.Label(label="Overall Score")

    with gr.Accordion("📋 Detailed Teacher Report", open=True):
        report_btn = gr.Button("Generate Table & Recommendation")
        report_display = gr.Markdown()

    btn.click(tutor_step, inputs=[audio_in], outputs=[audio_in, output_text, m_lab, q_image])
    report_btn.click(generate_parent_report, outputs=report_display)

demo.launch(allowed_paths=["/content/", "/content/seed/"])

/tmp/ipykernel_28514/3698593929.py:124: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e979e24127d2656bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Demo With multi-account management**

In [41]:
import gradio as gr
import numpy as np
import json
import os
from faster_whisper import WhisperModel

# ==========================================
# 1. SYSTEM & DATA MANAGEMENT
# ==========================================
class MwenyuraSystem:
    def __init__(self):
        self.data_dir = "/content/accounts/"
        if not os.path.exists(self.data_dir): os.makedirs(self.data_dir)

        self.current_user = "Guest"
        self.current_idx = 0
        self.skill_mastery = {"counting": 0.2, "addition": 0.2, "subtraction": 0.2, "word_problem": 0.2, "number_sense": 0.2}
        self.curriculum = self._load_curriculum()

    def _load_curriculum(self):
        path = '/content/seed/curriculum_seed.json'
        if os.path.exists(path):
            with open(path, 'r') as f: return json.load(f)
        return [{"stem_en": "Error: Load Seed File", "answer_int": 2, "skill": "counting", "visual": None}]

    def login(self, name):
        if not name: return "⚠️ Please enter a name!", "Muraho!", "0%", None
        self.current_user = name.strip().title()
        path = os.path.join(self.data_dir, f"{self.current_user.lower()}.json")

        if os.path.exists(path):
            with open(path, 'r') as f:
                data = json.load(f)
                self.skill_mastery = data["mastery"]
                self.current_idx = data["idx"]
            status = f"👋 **Welcome back, {self.current_user}!**"
        else:
            self.current_idx = 0
            self.skill_mastery = {k: 0.2 for k in self.skill_mastery}
            status = f"✨ **New account created for {self.current_user}!**"

        item = self.curriculum[self.current_idx]
        avg = f"{int(np.mean(list(self.skill_mastery.values())) * 100)}%"
        # NEW: Personalized greeting for the Learn window
        return status, f"Hi {self.current_user}! Muraho!\n\n{item['stem_en']}", avg, find_image(item.get('visual'))

    def save_progress(self):
        path = os.path.join(self.data_dir, f"{self.current_user.lower()}.json")
        with open(path, 'w') as f:
            json.dump({"mastery": self.skill_mastery, "idx": self.current_idx}, f)

system = MwenyuraSystem()
asr_model = WhisperModel("tiny.en", device="cpu", compute_type="int8")

# ==========================================
# 2. TUTOR & IMAGE LOGIC
# ==========================================
def find_image(visual_name):
    image_map = {"apples_3": "image_98439c.png", "goats_5": "image_97d300.png"}
    mapped = image_map.get(visual_name, visual_name)
    for loc in ["/content/seed/", "/content/"]:
        for ext in ['', '.png', '.jpg']:
            path = os.path.join(loc, f"{mapped}{ext}")
            if os.path.exists(path): return path
    return None

def tutor_step(audio_path):
    item = system.curriculum[system.current_idx]
    if not audio_path:
        return None, f"Hi {system.current_user}! 🔇 I didn't hear you!", f"{int(np.mean(list(system.skill_mastery.values()))*100)}%", find_image(item.get('visual'))

    segments, _ = asr_model.transcribe(audio_path)
    heard = " ".join([s.text for s in segments]).lower()

    is_correct = str(item["answer_int"]) in heard
    skill = item.get("skill", "counting")

    if is_correct:
        system.skill_mastery[skill] = min(0.99, system.skill_mastery[skill] + 0.15)
        if system.current_idx < len(system.curriculum) - 1:
            system.current_idx += 1

    system.save_progress()
    new_item = system.curriculum[system.current_idx]
    feedback = f"✅ Correct, {system.current_user}!\n\nNext: {new_item['stem_en']}" if is_correct else f"❌ Try again, {system.current_user}!\n\nQuestion: {item['stem_en']}"

    return None, feedback, f"{int(np.mean(list(system.skill_mastery.values())) * 100)}%", find_image(new_item.get('visual'))

# ==========================================
# 3. REPORT GENERATOR (GREEN & YELLOW STARS)
# ==========================================
def generate_fancy_report():
    avg_val = int(np.mean(list(system.skill_mastery.values())) * 100)
    best = max(system.skill_mastery, key=system.skill_mastery.get)
    weak = min(system.skill_mastery, key=system.skill_mastery.get)

    report = f"# 📊 Progress Report: {system.current_user}\n"
    report += f"**Overall Status:** {avg_val}% {'💚' * (avg_val // 20)}\n\n"
    report += "| Skill | Mastery Progress (5 Stars) |\n| :--- | :--- |\n"

    for s, v in system.skill_mastery.items():
        green_stars = max(1, int(v * 5))
        yellow_stars = 5 - green_stars
        star_row = ("💚" * green_stars) + ("💛" * yellow_stars)
        report += f"| {s.title().replace('_', ' ')} | {star_row} ({int(v*100)}%) |\n"

    tips = {
        "counting": "Use physical beans or stones to help visualize groups.",
        "addition": "Try adding spoons while setting the dinner table!",
        "subtraction": "Play 'hide and seek' with toys to practice taking away.",
        "word_problem": "Focus on identifying 'key words' like 'more' or 'left'.",
        "number_sense": "Look for numbers on bus signs and street doors."
    }

    report += f"\n---\n\n### 💡 Smart Recommendation\n\n"
    report += f"🌟 **Strongest Skill:** {best.title()}\n\n"
    report += f"📖 **Action Plan:** For **{weak.title()}**, {tips.get(weak)}\n\n"
    return report

# ==========================================
# 4. REFINED UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☀️ Mwenyura AI Math Tutor")

    with gr.Tabs() as tabs:
        with gr.Tab("🔘 Step 1: Login", id=0):
            u_input = gr.Textbox(label="Student Name")
            l_btn = gr.Button("Login / Create Account", variant="primary")
            l_status = gr.Markdown("Please log in to start.")

        with gr.Tab("🔘 Step 2: Learn", id=1):
            with gr.Row():
                with gr.Column():
                    q_img = gr.Image(label="Visual Aid")
                    audio_in = gr.Audio(sources="microphone", type="filepath")
                    sub_btn = gr.Button("Submit Answer", variant="primary")
                with gr.Column():
                    t_feedback = gr.Textbox(label="Tutor Feedback", value="Please login first...", lines=6)
                    m_label = gr.Label(label="Overall Mastery")

        with gr.Tab("🔘 Step 3: Progress Report", id=2):
            r_btn = gr.Button("Generate Report")
            r_md = gr.Markdown()

    # Event Handlers
    l_btn.click(system.login, inputs=u_input, outputs=[l_status, t_feedback, m_label, q_img])
    sub_btn.click(tutor_step, inputs=audio_in, outputs=[audio_in, t_feedback, m_label, q_img])
    r_btn.click(generate_fancy_report, outputs=r_md)

demo.launch(allowed_paths=["/content/seed/", "/content/"])

/tmp/ipykernel_28514/269005513.py:123: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10cdfe62d16dfcb179.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
